### Salary prediction, episode II: make it actually work (4 points)

Your main task is to use some of the tricks you've learned on the network and analyze if you can improve __validation MAE__. Try __at least 3 options__ from the list below for a passing grade. Write a short report about what you have tried. More ideas = more bonus points. 

__Please be serious:__ " plot learning curves in MAE/epoch, compare models based on optimal performance, test one change at a time. You know the drill :)

You can use either __pytorch__ or __tensorflow__ or any other framework (e.g. pure __keras__). Feel free to adapt the seminar code for your needs. For tensorflow version, consider `seminar_tf2.ipynb` as a starting point.


In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("./Train_rev1.zip", compression='zip', index_col=None)

In [3]:
data.sample(1)

,Id,Title,FullDescription,LocationRaw,LocationNormalized,ContractType,ContractTime,Company,Category,SalaryRaw,SalaryNormalized,SourceName
170510,71361318,Vehicle Paint Sprayer Middlesbrough,Vehicle Paint Sprayer MiddlesbroughAbout the ...,North Yorkshire - Middlesbrough,Middlesbrough,full_time,permanent,UKStaffsearch,Admin Jobs,20000 - 25000,22500,ukstaffsearch.com


In [4]:
text_columns = ["Title", "FullDescription"]
categorical_columns = ["Category", "Company", "LocationNormalized", "ContractType", "ContractTime"]
TARGET_COLUMN = "Log1pSalary"

data[categorical_columns] = data[categorical_columns].fillna('NaN') # cast missing values to string "NaN"

data.sample(3)

,Id,Title,FullDescription,LocationRaw,LocationNormalized,ContractType,ContractTime,Company,Category,SalaryRaw,SalaryNormalized,SourceName
170688,71362398,Electrical Field Service Engineer (Domestic Ap...,Electrical Field Service Engineer ( Water Heat...,City of London - London,The City,full_time,permanent,UKStaffsearch,Trade & Construction Jobs,24000 - 26000,25000,ukstaffsearch.com
83011,69043361,Network Support Engineer CCNP CCIP to ****K,"CCNP Service Provider NOC ENGINEER, CCNP Netwo...",Bristol Avon South West,UK,NaN,permanent,Imperative Recruitment,IT Jobs,32500 - 40000 per annum,36250,cwjobs.co.uk
104346,69572081,SUPPORT DEVELOPER,Support Developer London **** **** Company ...,London,London,NaN,permanent,Circle Recruitment,IT Jobs,30000.00 - 40000.00 GBP Annual,35000,jobserve.com


In [5]:
data['Log1pSalary'] = np.log1p(data['SalaryNormalized']).astype('float32')

In [6]:
import nltk
tokenizer = nltk.tokenize.WordPunctTokenizer()

def preprocess_text(text):
    text = str(text)
    return " ".join(tokenizer.tokenize(text.lower()))

data["Title"] = data["Title"].apply(preprocess_text)
data["FullDescription"] = data["FullDescription"].apply(preprocess_text)

In [7]:
from collections import Counter
token_counts = Counter()

# Count how many times does each token occur in both "Title" and "FullDescription" in total

for title in data["Title"].values:
    for token in title.split():
        token_counts[token] += 1
for descr in data["FullDescription"].values:
    for token in descr.split():
        token_counts[token] += 1


In [8]:
min_count = 10

# tokens from token_counts keys that had at least min_count occurrences throughout the dataset
tokens = sorted(token for token, count in token_counts.items() if count >= min_count)

# Add a special tokens for unknown and empty words
UNK, PAD = "UNK", "PAD"
tokens = [UNK, PAD] + tokens

In [9]:
token_to_id = {token:index for index, token in enumerate(tokens)}

In [10]:
UNK_IX, PAD_IX = map(token_to_id.get, [UNK, PAD])

def as_matrix(sequences, max_len=None):
    """ Convert a list of tokens into a matrix with padding """
    if isinstance(sequences[0], str):
        sequences = list(map(str.split, sequences))

    max_len = min(max(map(len, sequences)), max_len or float('inf'))

    matrix = np.full((len(sequences), max_len), np.int32(PAD_IX))
    for i,seq in enumerate(sequences):
        row_ix = [token_to_id.get(word, UNK_IX) for word in seq[:max_len]]
        matrix[i, :len(row_ix)] = row_ix

    return matrix

In [24]:
from sklearn.feature_extraction import DictVectorizer

# we only consider top-1k most frequent companies to minimize memory usage
top_companies, top_counts = zip(*Counter(data['Company']).most_common(1000))
recognized_companies = set(top_companies)
data["Company"] = data["Company"].apply(lambda comp: comp if comp in recognized_companies else "Other")

categorical_vectorizer = DictVectorizer(dtype=np.float32, sparse=False)
categorical_vectorizer.fit(data[categorical_columns].apply(dict, axis=1))

,dtype,<class 'numpy.float32'>
,separator,'='
,sparse,False
,sort,True


In [28]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(data, test_size=0.2, random_state=42)
data_train.index = range(len(data_train))
data_val.index = range(len(data_val))

print("Train size = ", len(data_train))
print("Validation size = ", len(data_val))

Train size =  195814
Validation size =  48954


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F


device = 'cuda' if torch.cuda.is_available() else 'cpu'


def to_tensors(batch, device):
    batch_tensors = dict()
    for key, arr in batch.items():
        if key in ["FullDescription", "Title"]:
            batch_tensors[key] = torch.tensor(arr, device=device, dtype=torch.int64)
        else:
            batch_tensors[key] = torch.tensor(arr, device=device)
    return batch_tensors


def make_batch(data, max_len=None, word_dropout=0, device=device):
    """
    Creates a keras-friendly dict from the batch data.
    :param word_dropout: replaces token index with UNK_IX with this probability
    :returns: a dict with {'title' : int64[batch, title_max_len]
    """
    batch = {}
    batch["Title"] = as_matrix(data["Title"].values, max_len)
    batch["FullDescription"] = as_matrix(data["FullDescription"].values, max_len)
    batch['Categorical'] = categorical_vectorizer.transform(data[categorical_columns].apply(dict, axis=1))

    if word_dropout != 0:
        batch["FullDescription"] = apply_word_dropout(batch["FullDescription"], 1. - word_dropout)

    if TARGET_COLUMN in data.columns:
        batch[TARGET_COLUMN] = data[TARGET_COLUMN].values

    return to_tensors(batch, device)

def apply_word_dropout(matrix, keep_prop, replace_with=UNK_IX, pad_ix=PAD_IX,):
    dropout_mask = np.random.choice(2, np.shape(matrix), p=[keep_prop, 1 - keep_prop])
    dropout_mask &= matrix != pad_ix
    return np.choose(dropout_mask, [matrix, np.full_like(matrix, replace_with)])

In [ ]:
import gensim
import gensim.downloader
word_vectors = gensim.downloader.load('glove-wiki-gigaword-100')
pretrained_weights = torch.FloatTensor(word_vectors.vectors)

In [31]:
class SalaryPredictor(nn.Module):
    def __init__(self, n_tokens, n_cat_features, pretrained_word_vectors, hid_size=128, freeze_embeddings=False):
        super().__init__()
        self.embedding_dim = pretrained_word_vectors.shape[1]
        self.hid_size = hid_size

        # Title processing branch
        self.title_emb = nn.Embedding.from_pretrained(pretrained_word_vectors, freeze=freeze_embeddings)
        self.title_lstm = nn.LSTM(self.embedding_dim, hid_size * 2, batch_first=True)
        self.title_ln = nn.LayerNorm(hid_size * 2)

        # Description processing branch
        self.descr_emb = nn.Embedding.from_pretrained(pretrained_word_vectors, freeze=freeze_embeddings)
        self.descr_lstm = nn.LSTM(self.embedding_dim, hid_size * 2, batch_first=True)
        self.desc_ln = nn.LayerNorm(hid_size * 2)

        # Categorical features processing branch
        self.cat_fn1 = nn.Linear(n_cat_features, n_cat_features * 2)
        self.bn1 = nn.BatchNorm1d(n_cat_features * 2)
        self.cat_fn2 = nn.Linear(n_cat_features * 2, n_cat_features)
        self.bn2 = nn.BatchNorm1d(n_cat_features)
        self.relu = nn.ReLU()

        # Combined features processing and final prediction
        combined_features_dim = hid_size * 2 + hid_size * 2 + n_cat_features
        self.all_fn1 = nn.Linear(combined_features_dim, combined_features_dim // 2)
        self.all_bn = nn.BatchNorm1d(combined_features_dim // 2)
        self.all_fn2 = nn.Linear(combined_features_dim // 2, 1)

    def forward(self, batch):
        # Process Title
        title = self.title_emb(batch["Title"])
        _, (title_h_n, _) = self.title_lstm(title)
        title = title_h_n.squeeze(0)
        title = self.title_ln(title)
        title = F.relu(title)

        # Process Description
        descr = self.descr_emb(batch["FullDescription"])
        _, (descr_h_n, _) = self.descr_lstm(descr)
        descr = descr_h_n.squeeze(0)
        descr = self.desc_ln(descr)
        descr = F.relu(descr)

        # Process Categorical
        cat = self.cat_fn1(batch["Categorical"])
        cat = self.bn1(cat)
        cat = self.relu(cat)

        cat = self.cat_fn2(cat)
        cat = self.bn2(cat)
        cat = self.relu(cat)

        # Process combined features
        combined = torch.cat([title, descr, cat], dim=1)
        combined = self.all_fn1(combined)
        combined = self.all_bn(combined)
        combined = self.relu(combined)

        prediction = self.all_fn2(combined)

        return torch.log1p(prediction).squeeze(1)

In [32]:
model = SalaryPredictor(len(tokens), len(categorical_vectorizer.feature_names_), pretrained_weights)

In [34]:
model = model.to(device)
batch = make_batch(data_train[:100], device=device)
criterion = nn.MSELoss()

dummy_pred = model(batch)
dummy_loss = criterion(dummy_pred, batch[TARGET_COLUMN])
assert dummy_pred.shape == torch.Size([100])
assert len(torch.unique(dummy_pred)) > 20, "model returns suspiciously few unique outputs. Check your initialization"
assert dummy_loss.ndim == 0 and 0. <= dummy_loss <= 250., "make sure you minimize MSE"

In [35]:
def iterate_minibatches(data, batch_size=256, shuffle=True, cycle=False, device=device, **kwargs):
    """ iterates minibatches of data in random order """
    while True:
        indices = np.arange(len(data))
        if shuffle:
            indices = np.random.permutation(indices)

        for start in range(0, len(indices), batch_size):
            batch = make_batch(data.iloc[indices[start : start + batch_size]], device=device, **kwargs)
            yield batch

        if not cycle: break

In [36]:
from tqdm.auto import tqdm

BATCH_SIZE = 64
EPOCHS = 100

In [37]:
def print_metrics(model, data, batch_size=BATCH_SIZE, name="", device=torch.device('cpu'), **kw):
    squared_error = abs_error = num_samples = 0.0
    model.eval()
    with torch.no_grad():
        for batch in iterate_minibatches(data, batch_size=batch_size, shuffle=False, device=device, **kw):
            batch_pred = model(batch)
            squared_error += torch.sum(torch.square(batch_pred - batch[TARGET_COLUMN]))
            abs_error += torch.sum(torch.abs(batch_pred - batch[TARGET_COLUMN]))
            num_samples += len(batch_pred)
    mse = squared_error.detach().cpu().numpy() / num_samples
    mae = abs_error.detach().cpu().numpy() / num_samples
    print("%s results:" % (name or ""))
    print("Mean square error: %.5f" % mse)
    print("Mean absolute error: %.5f" % mae)
    return mse, mae


In [39]:
criterion = nn.MSELoss(reduction='sum')
optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    print(f"epoch: {epoch}")
    model.train()
    for i, batch in tqdm(enumerate(
            iterate_minibatches(data_train, batch_size=BATCH_SIZE, device=device)),
            total=len(data_train) // BATCH_SIZE
        ):
        pred = model(batch)
        loss = criterion(pred, batch[TARGET_COLUMN])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print_metrics(model, data_val, device=device)



epoch: 0


  0%|          | 3/3059 [00:28<8:11:31,  9.65s/it]


KeyboardInterrupt: 

### A short report

Please tell us what you did and how did it work.

`<YOUR_TEXT_HERE>`, i guess...

## Recommended options

#### A) CNN architecture

All the tricks you know about dense and convolutional neural networks apply here as well.
* Dropout. Nuff said.
* Batch Norm. This time it's `nn.BatchNorm*`/`L.BatchNormalization`
* Parallel convolution layers. The idea is that you apply several nn.Conv1d to the same embeddings and concatenate output channels.
* More layers, more neurons, ya know...


#### B) Play with pooling

There's more than one way to perform pooling:
* Max over time (independently for each feature)
* Average over time (excluding PAD)
* Softmax-pooling:
$$ out_{i, t} = \sum_t {h_{i,t} \cdot {{e ^ {h_{i, t}}} \over \sum_\tau e ^ {h_{j, \tau}} } }$$

* Attentive pooling
$$ out_{i, t} = \sum_t {h_{i,t} \cdot Attn(h_t)}$$

, where $$ Attn(h_t) = {{e ^ {NN_{attn}(h_t)}} \over \sum_\tau e ^ {NN_{attn}(h_\tau)}}  $$
and $NN_{attn}$ is a dense layer.

The optimal score is usually achieved by concatenating several different poolings, including several attentive pooling with different $NN_{attn}$ (aka multi-headed attention).

The catch is that keras layers do not inlude those toys. You will have to [write your own keras layer](https://keras.io/layers/writing-your-own-keras-layers/). Or use pure tensorflow, it might even be easier :)

#### C) Fun with words

It's not always a good idea to train embeddings from scratch. Here's a few tricks:

* Use a pre-trained embeddings from `gensim.downloader.load`. See last lecture.
* Start with pre-trained embeddings, then fine-tune them with gradient descent. You may or may not download pre-trained embeddings from [here](http://nlp.stanford.edu/data/glove.6B.zip) and follow this [manual](https://keras.io/examples/nlp/pretrained_word_embeddings/) to initialize your Keras embedding layer with downloaded weights.
* Use the same embedding matrix in title and desc vectorizer


#### D) Going recurrent

We've already learned that recurrent networks can do cool stuff in sequence modelling. Turns out, they're not useless for classification as well. With some tricks of course..

* Like convolutional layers, LSTM should be pooled into a fixed-size vector with some of the poolings.
* Since you know all the text in advance, use bidirectional RNN
  * Run one LSTM from left to right
  * Run another in parallel from right to left 
  * Concatenate their output sequences along unit axis (dim=-1)

* It might be good idea to mix convolutions and recurrent layers differently for title and description


#### E) Optimizing seriously

* You don't necessarily need 100 epochs. Use early stopping. If you've never done this before, take a look at [early stopping callback(keras)](https://keras.io/callbacks/#earlystopping) or in [pytorch(lightning)](https://pytorch-lightning.readthedocs.io/en/latest/common/early_stopping.html).
  * In short, train until you notice that validation
  * Maintain the best-on-validation snapshot via `model.save(file_name)`
  * Plotting learning curves is usually a good idea
  
Good luck! And may the force be with you!